# Wildfire Prediction — Ensemble Model
Random Forest + Extra Trees + Gradient Boosting + Logistic Regression + Cox PH Survival

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

In [ ]:
train = pd.read_csv("content/FE_train.csv")
test  = pd.read_csv("content/FE_test.csv")

with open("feature_selection/final_feature_set.json") as f:
    selected_features = json.load(f)["final_features"]

print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Selected features: {len(selected_features)}")

In [ ]:
def add_features(df):
    df = df.copy()
    # Interaction: fire growth x spread speed
    df["growth_x_speed"]    = df["radial_growth_m"] * df["centroid_speed_m_per_h"]
    # How far the fire is relative to its closing speed (estimated time-to-breach)
    df["dist_over_closing"] = df["dist_min_ci_0_5h"] / (df["closing_speed_abs_m_per_h"] + 1e-6)
    # Growth quality weighted by trajectory fit quality
    df["growth_x_fit"]      = df["area_growth_rel_0_5h"] * df["dist_fit_r2_0_5h"]
    # Bearing alignment x speed (how directly fire heads toward zone)
    df["aligned_speed"]     = df["alignment_abs"] * df["centroid_speed_m_per_h"]
    # Rate of change of distance vs current distance
    df["dist_trend_ratio"]  = df["dist_slope_ci_0_5h"] / (df["dist_min_ci_0_5h"] + 1e-6)
    # Log of estimated breach time
    df["log_time_to_close"] = np.log1p(df["dist_over_closing"].clip(0))
    return df

X_train = add_features(train[selected_features])
X_test  = add_features(test[selected_features])
print(f"Total features after engineering: {X_train.shape[1]}")

In [ ]:
# Create binary label for each time horizon:
# label=1 if fire HIT within H hours, 0 otherwise (censored OR hit later)
horizons = [12, 24, 48, 72]

for H in horizons:
    train[f"hit_{H}h"] = ((train["event"] == 1) & (train["time_to_hit_hours"] <= H)).astype(int)

print("Label distributions:")
for H in horizons:
    print(f"  {H}h: {train[f'hit_{H}h'].mean():.3f} positive rate  (n={train[f'hit_{H}h'].sum()})")

In [ ]:
class CoxPHSurvival:
    """
    Cox Proportional Hazards model with Breslow baseline estimator.
    Implemented using scipy L-BFGS-B (no extra packages required).
    alpha: L2 regularisation strength
    """
    def __init__(self, alpha=0.5):
        self.alpha  = alpha
        self.coef_  = None
        self.scaler = StandardScaler()

    def _partial_loglik(self, beta, X, time, event):
        """Negative partial log-likelihood + L2 penalty."""
        Xb    = X @ beta
        order = np.argsort(-time)
        Xb_o  = Xb[order]
        ev_o  = event[order]
        log_lik = 0.0
        log_cum = np.logaddexp.accumulate(Xb_o)
        for i in range(len(Xb_o)):
            if ev_o[i]:
                log_lik += Xb_o[i] - log_cum[i]
        return -(log_lik - 0.5 * self.alpha * np.dot(beta, beta))

    def _grad(self, beta, X, time, event):
        """Gradient of negative partial log-likelihood."""
        Xb    = X @ beta
        order = np.argsort(-time)
        X_o, Xb_o, ev_o = X[order], Xb[order], event[order]
        grad = np.zeros_like(beta)
        log_sum, sum_expX = -np.inf, np.zeros(X.shape[1])
        for i in range(len(Xb_o)):
            log_sum  = np.logaddexp(log_sum, Xb_o[i])
            sum_expX += np.exp(Xb_o[i]) * X_o[i]
            if ev_o[i]:
                grad += sum_expX / np.exp(log_sum) - X_o[i]
        return grad + self.alpha * beta

    def fit(self, X, time, event):
        Xs = self.scaler.fit_transform(X)
        res = minimize(self._partial_loglik, np.zeros(Xs.shape[1]),
                       args=(Xs, time, event), jac=self._grad,
                       method="L-BFGS-B", options={"maxiter": 500})
        self.coef_ = res.x
        self._fit_baseline(Xs, time, event)
        return self

    def _fit_baseline(self, Xs, time, event):
        """Breslow estimator for cumulative baseline hazard H0(t)."""
        Xb    = Xs @ self.coef_
        order = np.argsort(time)
        t_o, ev_o, Xb_o = time[order], event[order], Xb[order]
        times, h0 = [], []
        for i in range(len(t_o)):
            if ev_o[i]:
                times.append(t_o[i])
                h0.append(1.0 / (np.sum(np.exp(Xb_o[i:])) + 1e-12))
        self.baseline_times_ = np.array(times)
        self.baseline_H0_    = np.cumsum(h0)

    def predict_hit_proba(self, X, t):
        """P(T <= t | X) for each sample."""
        Xs   = self.scaler.transform(X)
        Xb   = Xs @ self.coef_
        idx  = np.searchsorted(self.baseline_times_, t, side="right") - 1
        H0_t = self.baseline_H0_[idx] if idx >= 0 else 0.0
        surv = np.exp(-H0_t * np.exp(Xb))
        return 1.0 - surv

print("CoxPHSurvival class defined.")

In [ ]:
print("Fitting Cox PH model (uses full survival structure)...")

cox = CoxPHSurvival(alpha=0.5)
cox.fit(X_train.values, train["time_to_hit_hours"].values, train["event"].values)
print("Cox PH fitted.")

max_time = train["time_to_hit_hours"].max()  # 66.99h — cap for 72h horizon

cox_preds = {}
for H in horizons:
    t = min(H, max_time)
    cox_preds[H] = cox.predict_hit_proba(X_test.values, t)
    print(f"  Cox P(hit<={H}h): mean={cox_preds[H].mean():.3f}")

In [ ]:
def build_ensemble(X, y, X_test_data, seed=42):
    """Train RF + ET + GB + LR ensemble for one horizon label."""
    models = [
        ("RF", RandomForestClassifier(
            n_estimators=500, min_samples_leaf=2, min_samples_split=5,
            max_features="sqrt", class_weight="balanced",
            random_state=seed, n_jobs=-1)),
        ("ET", ExtraTreesClassifier(
            n_estimators=500, min_samples_leaf=2, min_samples_split=5,
            max_features="sqrt", class_weight="balanced",
            random_state=seed+1, n_jobs=-1)),
        ("GB", GradientBoostingClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            min_samples_leaf=3, subsample=0.8,
            max_features="sqrt", random_state=seed+2)),
        ("LR", Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(C=0.1, class_weight="balanced",
                                           max_iter=1000, random_state=seed+3))])),
    ]
    skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    preds = []
    for name, model in models:
        oof = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
        print(f"    {name}: OOF AUC = {roc_auc_score(y, oof):.4f}")
        model.fit(X, y)
        preds.append(model.predict_proba(X_test_data)[:, 1])
    return np.mean(preds, axis=0)

print("Ensemble function defined.")

In [ ]:
ensemble_preds = {}
for H in horizons:
    print(f"Horizon {H}h ---")
    ensemble_preds[H] = build_ensemble(
        X_train.values,
        train[f"hit_{H}h"].values,
        X_test.values
    )

In [ ]:
# Blend: 30% Cox PH (survival structure) + 70% Ensemble (non-linear patterns)
COX_WEIGHT = 0.30

blended = {H: COX_WEIGHT * cox_preds[H] + (1 - COX_WEIGHT) * ensemble_preds[H]
           for H in horizons}

# Enforce monotonicity: P(12h) <= P(24h) <= P(48h) <= P(72h)
p12 = np.clip(blended[12], 0, 1)
p24 = np.clip(np.maximum(blended[24], p12), 0, 1)
p48 = np.clip(np.maximum(blended[48], p24), 0, 1)
p72 = np.clip(np.maximum(blended[72], p48), 0, 1)

print("Monotonicity check (all should be True):")
print(f"  P(12h) <= P(24h): {np.all(p24 >= p12 - 1e-9)}")
print(f"  P(24h) <= P(48h): {np.all(p48 >= p24 - 1e-9)}")
print(f"  P(48h) <= P(72h): {np.all(p72 >= p48 - 1e-9)}")

In [ ]:
submission = pd.DataFrame({
    "event_id": test["event_id"],
    "prob_12h":  p12,
    "prob_24h":  p24,
    "prob_48h":  p48,
    "prob_72h":  p72
})

submission.to_csv("content/submission_final.csv", index=False)
print("Saved to content/submission_final.csv")
print(submission.head(10).to_string(index=False))

print("Stats:")
for col in ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]:
    s = submission[col]
    print(f"  {col}: mean={s.mean():.3f}  std={s.std():.3f}  min={s.min():.3f}  max={s.max():.3f}")